1. 문서의 내용을 읽는다
2. 문서를 쪼갠다.
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을 대, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [20]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

loader = Docx2txtLoader("./tax.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [21]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_chroma import Chroma 
# database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax' ,persist_directory="./chroma_db")
database = Chroma(collection_name='chroma-tax', persist_directory="./chroma_db", embedding_function=embedding)

In [25]:
query = '연봉 5000만원인 직장인의 소득세는 얼마인가요?'
# retrieve_docs = database.similarity_search(query=query, k=3)

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")

In [27]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

In [18]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [32]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)


In [33]:
ai_message = qa_chain({"query": query})

In [34]:
ai_message

{'query': '연봉 5000만원인 직장인의 소득세는 얼마인가요?',
 'result': '이 자료에는 소득세율표나 근로소득 공제 같은 구체적인 세액 계산 정보가 포함되어 있지 않아 연봉 5,000만원의 정확한 소득세를 계산할 수 없습니다. 소득세액은 거주자 여부, 인적공제, 근로소득공제 등 다양한 공제에 따라 달라집니다. 따라서 최신 세율표와 본인의 공제 상황이 필요합니다.'}